In [ ]:
from collections import defaultdict

import os

import json
import numpy as np
import pandas as pd
import pickle
import polars as pl

from transformers import LlamaModel, LlamaTokenizer

import torch
from torch.utils.data import DataLoader

from tqdm import tqdm as tqdm

In [ ]:
DATA_DIR = '../data'

NUM_PARTS = 10

DATASET_PATH = f'{DATA_DIR}/beauty/raw'
OUTPUT_DIR = f'{DATA_DIR}/beauty'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_DATA_PATH = f'{DATASET_PATH}/Beauty_5.json'
METADATA_PATH = f'{DATASET_PATH}/metadata.json'
DATASET_METADATA_PATH = f'{DATASET_PATH}/content_embeddings.parquet'

In [ ]:
item_ids = []
user_ids = []
timestamps = []

user_id_mapping = {}
with open(RAW_DATA_PATH, 'r') as f:
    for line in f.readlines():
        row = json.loads(line)
        item_ids.append(row['asin'])
        user_ids.append(row['reviewerID'])
        timestamps.append(row['unixReviewTime'])

        if row['reviewerID'] not in user_id_mapping:
            user_id_mapping[row['reviewerID']] = len(user_id_mapping)


interactions_df = pl.DataFrame({
    'item_id': item_ids,
    'user_id': user_ids,
    'timestamp': timestamps
})

user_id_mapping_df = pl.DataFrame({
    'old_user_id': list(user_id_mapping.keys()),
    'new_user_id': list(user_id_mapping.values())
})

interactions_df.head()

In [ ]:
filtering_stage = 0
is_changed = True
threshold = 5
good_users = set()
good_items = set()

filtered_df = interactions_df.clone()

while is_changed:
    user_counts = filtered_df.group_by('user_id').agg(
        pl.len().alias('user_count'),
    )
    item_counts = filtered_df.group_by('item_id').agg(
        pl.len().alias('item_count'),
    )

    good_users = user_counts.filter(pl.col('user_count') >= threshold).select(
        'user_id',
    )
    good_items = item_counts.filter(pl.col('item_count') >= threshold).select(
        'item_id',
    )

    old_size = len(filtered_df)

    new_df = filtered_df.join(good_users, on='user_id', how='inner')
    new_df = new_df.join(good_items, on='item_id', how='inner')

    new_size = len(new_df)

    filtered_df = new_df
    is_changed = old_size != new_size
    filtering_stage += 1


filtered_df = filtered_df.sort(['timestamp']).with_row_index('original_order')
all_data_interactions = filtered_df.clone()
all_data_interactions.head()

In [ ]:
user_mapping = all_data_interactions.select(pl.col('user_id')).unique().sort('user_id').with_row_index('new_user_id')

all_data_interactions = (
    all_data_interactions
    .join(user_mapping, on="user_id", how="left")
    .with_columns(pl.col("new_user_id").cast(pl.Int64).alias("user_id"))
    .drop("new_user_id")
)
all_data_interactions.head()

In [ ]:
all_data_interactions.shape

In [ ]:
all_data_items = all_data_interactions.select('item_id').unique()
all_data_users = all_data_interactions.select('user_id').unique()

unique_items_sorted = all_data_items.sort('item_id').with_row_index('new_item_id')
global_item_mapping = dict(zip(unique_items_sorted['item_id'], unique_items_sorted['new_item_id']))

print(f"Total users: {all_data_users.shape[0]}, Total items: {len(global_item_mapping)}")

In [ ]:
mapping_output_path = f"{OUTPUT_DIR}/global_item_mapping.json"

with open(mapping_output_path, 'w') as f:
    json.dump({str(k): v for k, v in global_item_mapping.items()}, f, indent=2)

print(f"Mapping saved: {mapping_output_path}")

In [ ]:
valid_asin = list(set(global_item_mapping.keys()))
valid_asin

In [ ]:
def getDF(path):
    i = 0
    df = {}
    with open(path, 'r') as f:
        for line in f.readlines():
            df[i] = eval(line)
            i += 1

    return pd.DataFrame.from_dict(df, orient="index")

metadata_df = getDF(METADATA_PATH)
metadata_df = pl.from_pandas(metadata_df)
metadata_df.head()

In [ ]:
def preprocess(row: pd.Series):
    row = row.fillna("None")
    return f"Title: {row['title']}. Categories: {', '.join(row['categories'][0])}. Description: {row['description']}."


def get_data(metadata_df, valid_asin):
    filtered_df = (
        metadata_df
            .filter(pl.col('asin').is_in(valid_asin))
            .select(pl.col('asin'), pl.col('title'), pl.col('description'), pl.col('categories'))
    )
    filtered_df = filtered_df.to_pandas()
    filtered_df["combined_text"] = filtered_df.apply(preprocess, axis=1)

    return filtered_df


In [ ]:
metadata_textual_df = pl.from_pandas(get_data(metadata_df, valid_asin))
metadata_textual_df.head()

In [ ]:
device = torch.device('cuda:0')

model_name = "huggyllama/llama-7b"
tokenizer = LlamaTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = LlamaModel.from_pretrained(model_name)
model = model.to(device)
model = model.eval()


class MyDataset:
    def __init__(self, data: pl.DataFrame):
        self._data = []
        for row in data.iter_rows(named=True):
            self._data.append({
                'asin': row['asin'],
                'text': row['combined_text']
            })

    def __len__(self):
        return len(self._data)

    def __getitem__(self, idx):
        text = self._data[idx]['text']
        inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True, padding="max_length")
        return {
            'asin': self._data[idx]['asin'],
            'input_ids': inputs['input_ids'][0],
            'attention_mask': inputs['attention_mask'][0]
        }
    

dataset = MyDataset(metadata_textual_df)
loader = DataLoader(
    dataset, 
    batch_size=15, 
    drop_last=False, 
    shuffle=False, 
    num_workers=4,
    pin_memory=True
)

metadata_item_ids = []
metadata_item_embeddings = []

for batch in tqdm(loader):
    with torch.inference_mode():
        outputs = model(
            input_ids=batch["input_ids"].to(device), 
            attention_mask=batch["attention_mask"].to(device)
        )
        embeddings = outputs.last_hidden_state
    
        embeddings = outputs.last_hidden_state  # (bs, sl, ed)
        embeddings[(~batch["attention_mask"].bool())] = 0. # (bs, sl, ed)

    metadata_item_ids += batch['asin']
    metadata_item_embeddings += embeddings.mean(dim=1).tolist()  # (bs, ed)

In [ ]:
items_metadata = pl.DataFrame({
    'item_id': metadata_item_ids,
    'embedding': metadata_item_embeddings
})

items_metadata.write_parquet(DATASET_METADATA_PATH)
items_metadata.head()

In [ ]:
def remap_interactions(df, mapping): 
    return df.with_columns( 
        pl.col('item_id') 
        .map_elements(lambda x: mapping.get(x, None), return_dtype=pl.UInt32)
    ) 

all_data_interactions_remapped = remap_interactions(all_data_interactions, global_item_mapping) 
items_metadata_remapped = remap_interactions(items_metadata, global_item_mapping)

In [ ]:
NUM_PARTS = 10

all_data_interactions_remapped_sorted = all_data_interactions_remapped.sort("original_order")
all_data_interactions_remapped_sorted = all_data_interactions_remapped_sorted.with_row_index('row_nr')

base_size = all_data_interactions_remapped_sorted.height // NUM_PARTS

all_data_interactions_with_groups= all_data_interactions_remapped_sorted.with_columns(
    (pl.col("row_nr") // (base_size + 1)).alias("part")
).drop("row_nr")

In [ ]:
all_data_interactions_with_groups.head()

In [ ]:
parts_distribution = all_data_interactions_with_groups.group_by("part").agg(
    pl.count().alias("count"),
    pl.col("original_order").min().alias("min_order"),
    pl.col("original_order").max().alias("max_order")
).sort("part")

print("\nDistribution by part:")
print(parts_distribution)

print(f"Minimum part: {parts_distribution['part'].min()}")
print(f"Maximum part: {parts_distribution['part'].max()}")
print(f"Total number of events from all parts: {parts_distribution['count'].sum()}")

In [ ]:
def write_parquet(output_dir, data, file_name):
    print(f"Shape: {data.shape}")
    output_parquet_path = f"{output_dir}/{file_name}.parquet"
    data.write_parquet(output_parquet_path)
    print(f"File saved: {file_name}")

write_parquet(OUTPUT_DIR, items_metadata_remapped, "items_metadata_remapped")
write_parquet(OUTPUT_DIR, all_data_interactions_with_groups, "all_data_interactions_with_groups")